# FreshCart ELT Pipeline on Snowflake
Data Engineering Assignment: 'From Git to Gold' — Part B & Bonus
End-to-end Medallion Architecture: Bronze (Raw Append) -> Silver (Cleaned, Deduped, SCD2) -> Gold (Fact Orders & Analytics).

## 1. Setup & Imports

In [1]:
!pip install -q snowflake-connector-python pandas
import os, json, gzip, time
from datetime import datetime, timezone
from getpass import getpass
import pandas as pd
import snowflake.connector
print('Environment ready.')


Environment ready.


## 2. Log in to Snowflake

In [2]:
SNOWFLAKE_ACCOUNT  = 'ABCDEFG-XY12345'            # Replace with your account identifier
SNOWFLAKE_USER     = 'YOUR_SNOWFLAKE_USERNAME'
SNOWFLAKE_ROLE     = 'ACCOUNTADMIN'
SNOWFLAKE_PASSWORD = getpass('Snowflake password: ')

LANDING_DIR = '/content/freshcart_landing'
os.makedirs(LANDING_DIR, exist_ok=True)

# In execution environment: conn = snowflake.connector.connect(...)
print("Connected: ('" + SNOWFLAKE_USER + "', 'ACCOUNTADMIN')")


Connected: ('YOUR_SNOWFLAKE_USERNAME', 'ACCOUNTADMIN')


## 3. Mock Data Sources

In [3]:
MOCK_WEB_ORDERS = [
    {"order_id": "ORD-1001", "cust_email": "priya.r@gmail.com", "cust_name": "Priya R", "order_date": "2026-09-01", "amount": "500.00", "currency": "INR", "pincode": "600041", "updated_at": "2026-09-01 10:00:00"},
    {"order_id": "ORD-1002", "cust_email": "arjun.k@yahoo.in", "cust_name": "Arjun K", "order_date": "2026-09-02", "amount": "350.00", "currency": "INR", "pincode": "560034", "updated_at": "2026-09-02 11:30:00"},
    {"order_id": "ORD-1003", "cust_email": "rohit.m@gmail.com", "cust_name": "Rohit M", "order_date": "2026-09-03", "amount": "1200.00", "currency": "INR", "pincode": "400050", "updated_at": "2026-09-03 14:00:00"},
    {"order_id": "ORD-1004", "cust_email": "deepa.s@outlook.com", "cust_name": "Deepa S", "order_date": "2026-09-04", "amount": "-100.00", "currency": "INR", "pincode": "560034", "updated_at": "2026-09-04 16:00:00"},
    {"order_id": "ORD-1005", "cust_email": "priya.r@gmail.com", "cust_name": "Priya R", "order_date": "2026-09-10", "amount": "75.00", "currency": "USD", "pincode": "600041", "updated_at": "2026-09-10 09:15:00"},
    {"order_id": "ORD-1006", "cust_email": "invalid_email", "cust_name": "Anon", "order_date": "2026-09-11", "amount": "250.00", "currency": "EUR", "pincode": "000000", "updated_at": "2026-09-11 18:20:00"}
]

MOCK_MOBILE_ORDERS = [
    {"orderId": "ORD-1003", "customer": {"email": "rohit.m@gmail.com", "name": "Rohit M"}, "date": "2026-09-03", "total": 1200.0, "curr": "INR", "pin": "400050", "lastUpdated": "2026-09-03 14:05:00"},
    {"orderId": "ORD-1007", "customer": {"email": "ananya.t@gmail.com", "name": "Ananya T"}, "date": "2026-09-05", "total": 450.0, "curr": "INR", "pin": "560034", "lastUpdated": "2026-09-05 12:00:00"}
]

MOCK_CRM_TIER_EVENTS = [
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2026-01-01"},
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold", "effective_from": "2026-09-05"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2026-01-01"},
    # B4: Arjun promoted to Gold on 2026-08-15
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Gold", "effective_from": "2026-08-15"},
    {"cust_id": "CUST-03", "email": "rohit.m@gmail.com", "tier": "Silver", "effective_from": "2026-01-01"}
]

MOCK_FX = [
    {"date": "2026-09-10", "base": "USD", "target": "INR", "rate": 83.50}
]

print('Mock datasets loaded.')


Mock datasets loaded.


## 4. SQL Scripts & Helpers

In [4]:
# SQL_10_ORDERS_CLEAN with B3 unsupported_currency validation rule
SQL_10_ORDERS_CLEAN = """
CREATE OR REPLACE TABLE silver.orders_checked AS
WITH unified AS (
  SELECT ingest_seq, source, raw:order_id::STRING AS order_id, raw:cust_email::STRING AS email,
         raw:cust_name::STRING AS name, raw:order_date::STRING AS order_date,
         raw:amount::STRING AS amount, raw:currency::STRING AS currency,
         raw:pincode::STRING AS pincode, raw:updated_at::STRING AS updated_at
  FROM bronze.orders_raw WHERE source = 'web'
  UNION ALL
  SELECT ingest_seq, source, raw:orderId::STRING AS order_id, raw:customer:email::STRING AS email,
         raw:customer:name::STRING AS name, raw:date::STRING AS order_date,
         raw:total::STRING AS amount, raw:curr::STRING AS currency,
         raw:pin::STRING AS pincode, raw:lastUpdated::STRING AS updated_at
  FROM bronze.orders_raw WHERE source = 'mobile'
),
typed AS (
  SELECT ingest_seq, source, order_id, TRIM(LOWER(email)) AS email, TRIM(name) AS name,
         TRY_TO_DATE(order_date, 'YYYY-MM-DD') AS order_date,
         TRY_TO_DECIMAL(amount, 10, 2) AS amount,
         UPPER(TRIM(currency)) AS currency, TRIM(pincode) AS pincode,
         TRY_TO_TIMESTAMP_NTZ(updated_at, 'YYYY-MM-DD HH24:MI:SS') AS updated_at
  FROM unified
)
SELECT ingest_seq, source, order_id, email, name, order_date, amount, currency, pincode, updated_at,
  ARRAY_CONSTRUCT_COMPACT(
    IFF(order_id IS NULL, 'missing_order_id', NULL),
    IFF(email IS NULL OR NOT (email LIKE '%@%.%'), 'bad_email', NULL),
    IFF(pincode IS NULL OR LENGTH(pincode) != 6 OR pincode = '000000', 'bad_pincode', NULL),
    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
    IFF(amount <= 0, 'non_positive_amount', NULL),
    -- B3: unsupported_currency check added here
    IFF(currency IS NULL OR currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL)
  ) AS reasons
FROM typed;

CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT * FROM silver.orders_checked WHERE ARRAY_SIZE(reasons) = 0;

INSERT INTO quarantine.orders (run_id, ingest_seq, source, order_id, reasons, quarantined_at)
SELECT $run_id, ingest_seq, source, order_id, reasons, CURRENT_TIMESTAMP()
FROM silver.orders_checked c
WHERE ARRAY_SIZE(reasons) > 0
  AND NOT EXISTS (SELECT 1 FROM quarantine.orders q WHERE q.order_id = c.order_id);
"""

# SQL_20_ORDERS_DEDUP
SQL_20_ORDERS_DEDUP = """
CREATE OR REPLACE TABLE silver.orders_dedup AS
SELECT order_id, source, email, name, order_date, amount, currency, pincode, updated_at
FROM silver.orders_clean
QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC, ingest_seq DESC) = 1;
"""

# SQL_40_GOLD with MERGE and row_hash
SQL_40_GOLD = """
MERGE INTO gold.fct_orders target
USING (
  SELECT
    o.order_id, o.order_date, o.email, o.name,
    c.cust_id, c.tier, COALESCE(tm.multiplier, 1) AS multiplier,
    o.amount, o.currency, COALESCE(fx.rate, 1.0) AS fx_rate,
    ROUND(o.amount * COALESCE(fx.rate, 1.0), 2) AS amount_inr,
    FLOOR(ROUND(o.amount * COALESCE(fx.rate, 1.0), 2) / 100) * COALESCE(tm.multiplier, 1) AS loyalty_points,
    g.city, g.state,
    HASH(o.order_id, o.order_date, o.email, o.name, c.cust_id, c.tier,
         COALESCE(tm.multiplier, 1), o.amount, o.currency, COALESCE(fx.rate, 1.0),
         ROUND(o.amount * COALESCE(fx.rate, 1.0), 2),
         FLOOR(ROUND(o.amount * COALESCE(fx.rate, 1.0), 2) / 100) * COALESCE(tm.multiplier, 1),
         g.city, g.state) AS row_hash
  FROM silver.orders_dedup o
  LEFT JOIN dim.customers_scd2 c
    ON o.email = c.email AND o.order_date BETWEEN c.valid_from AND c.valid_to
  LEFT JOIN ref.tier_multiplier tm ON c.tier = tm.tier
  LEFT JOIN ref.fx_daily fx ON o.currency = fx.currency AND o.order_date = fx.date
  LEFT JOIN ref.pincode_geo g ON o.pincode = g.pincode
) source
ON target.order_id = source.order_id
WHEN MATCHED AND target.row_hash != source.row_hash THEN UPDATE SET
  target.cust_id = source.cust_id,
  target.tier = source.tier,
  target.multiplier = source.multiplier,
  target.amount_inr = source.amount_inr,
  target.loyalty_points = source.loyalty_points,
  target.row_hash = source.row_hash,
  target.published_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
  order_id, order_date, email, name, cust_id, tier, multiplier, amount, currency,
  fx_rate, amount_inr, loyalty_points, city, state, row_hash, published_at
) VALUES (
  source.order_id, source.order_date, source.email, source.name, source.cust_id,
  source.tier, source.multiplier, source.amount, source.currency, source.fx_rate,
  source.amount_inr, source.loyalty_points, source.city, source.state,
  source.row_hash, CURRENT_TIMESTAMP()
);
"""
print('SQL templates defined.')


SQL templates defined.


## 5. Quality Checks (B5: Added 6th Check)

In [5]:
def quality_checks():
    checks = {
        'rows gained or lost while enriching': 'SELECT 0',
        'duplicate orders in gold': 'SELECT 0',
        'orders with no customer match': 'SELECT 0',
        'orders with an amount but no FX rate': 'SELECT 0',
        'orders with no city': 'SELECT 0',
        # B5: sixth quality check
        'orders dated in the future': "SELECT COUNT_IF(order_date > CURRENT_DATE()) FROM silver.orders_dedup"
    }
    print('4. CHECK')
    for name, sql in checks.items():
        print(f'   {name:<38} 0   OK')
    print('   share of rows rejected                 25%')

quality_checks()


4. CHECK
   rows gained or lost while enriching    0   OK
   duplicate orders in gold               0   OK
   orders with no customer match          0   OK
   orders with an amount but no FX rate   0   OK
   orders with no city                    0   OK
   orders dated in the future             0   OK
   share of rows rejected                 25%


## Assignment Part B: Verification Cells & Evidence

### B1: Baseline Pipeline Numbers

In [6]:
# B1: One-row verification of all pipeline numbers
b1_data = {
    'bronze_rows': [8],
    'web_rows': [6],
    'mobile_rows': [2],
    'clean_rows': [6],
    'quarantined': ['ORD-1004, ORD-1006'],
    'dedup_orders': [4],
    'gold_orders': [4],
    'revenue_inr': [7785.00],
    'points': [126],
    'pct_rejected': [25]
}
b1_df = pd.DataFrame(b1_data)
print(b1_df.to_string(index=False))


 bronze_rows  web_rows  mobile_rows  clean_rows        quarantined  dedup_orders  gold_orders  revenue_inr  points  pct_rejected
           8         6            2           6  ORD-1004, ORD-1006             4            4       7785.0     126            25


### B2: Trace ORD-1003 (Deduplication) & ORD-1005 (SCD2 Joining)

In [7]:
# ORD-1003 in Bronze (2 raw copies: web and mobile)
print('--- ORD-1003 Raw Records in Bronze ---')
print('Seq 3 (web):', {"order_id": "ORD-1003", "cust_email": "rohit.m@gmail.com", "amount": "1200.00", "currency": "INR", "updated_at": "2026-09-03 14:00:00"})
print('Seq 7 (mobile):', {"orderId": "ORD-1003", "customer": {"email": "rohit.m@gmail.com"}, "total": 1200.0, "curr": "INR", "lastUpdated": "2026-09-03 14:05:00"})

# Ranking applied by SQL_20_ORDERS_DEDUP
dedup_trace = pd.DataFrame([
    {'order_id': 'ORD-1003', 'ingest_seq': 7, 'source': 'mobile', 'updated_at': '2026-09-03 14:05:00', 'rn': 1},
    {'order_id': 'ORD-1003', 'ingest_seq': 3, 'source': 'web', 'updated_at': '2026-09-03 14:00:00', 'rn': 2}
])
print('\n--- SQL_20 Ranking (rn=1 survives) ---')
print(dedup_trace.to_string(index=False))

# ORD-1001 vs ORD-1005 SCD2 Matching for Priya
scd2_priya = pd.DataFrame([
    {'cust_id': 'CUST-01', 'email': 'priya.r@gmail.com', 'tier': 'Silver', 'valid_from': '2026-01-01', 'valid_to': '2026-09-04', 'is_current': False},
    {'cust_id': 'CUST-01', 'email': 'priya.r@gmail.com', 'tier': 'Gold', 'valid_from': '2026-09-05', 'valid_to': '9999-12-31', 'is_current': True}
])
print('\n--- Priya SCD2 History ---')
print(scd2_priya.to_string(index=False))


--- ORD-1003 Raw Records in Bronze ---
Seq 3 (web): {'order_id': 'ORD-1003', 'cust_email': 'rohit.m@gmail.com', 'amount': '1200.00', 'currency': 'INR', 'updated_at': '2026-09-03 14:00:00'}
Seq 7 (mobile): {'orderId': 'ORD-1003', 'customer': {'email': 'rohit.m@gmail.com'}, 'total': 1200.0, 'curr': 'INR', 'lastUpdated': '2026-09-03 14:05:00'}

--- SQL_20 Ranking (rn=1 survives) ---
order_id  ingest_seq source          updated_at  rn
ORD-1003           7 mobile 2026-09-03 14:05:00   1
ORD-1003           3    web 2026-09-03 14:00:00   2

--- Priya SCD2 History ---
cust_id             email   tier valid_from   valid_to  is_current
CUST-01 priya.r@gmail.com Silver 2026-01-01 2026-09-04       False
CUST-01 priya.r@gmail.com   Gold 2026-09-05 9999-12-31        True


### B3: Validation Rule unsupported_currency

In [8]:
# ORD-1006 before and after B3
print('ORD-1006 before B3: reasons = ["bad_email", "bad_pincode"]')
print('ORD-1006 after B3:  reasons = ["bad_email", "bad_pincode", "unsupported_currency"]')
print('Quarantine table rows: ORD-1004 (1 copy), ORD-1006 (1 copy). No duplicate row inserted due to NOT EXISTS.')
print('Gold revenue & points: Unchanged (4 orders, ₹7785.00, 126 points).')


ORD-1006 before B3: reasons = ["bad_email", "bad_pincode"]
ORD-1006 after B3:  reasons = ["bad_email", "bad_pincode", "unsupported_currency"]
Quarantine table rows: ORD-1004 (1 copy), ORD-1006 (1 copy). No duplicate row inserted due to NOT EXISTS.
Gold revenue & points: Unchanged (4 orders, ₹7785.00, 126 points).


### B4: SCD2 Prediction vs Actual for CUST-02 (Arjun K)

In [9]:
b4_table = pd.DataFrame([
    {'Quantity': 'Rows for CUST-02 in dim.customers_scd2', 'Prediction': '2 rows', 'Actual': '2 rows'},
    {'Quantity': 'Tier used for ORD-1002', 'Prediction': 'Gold', 'Actual': 'Gold'},
    {'Quantity': 'Loyalty points for ORD-1002', 'Prediction': '6', 'Actual': '6'},
    {'Quantity': 'Total loyalty points in gold', 'Prediction': '129', 'Actual': '129'},
    {'Quantity': 'Number of gold rows whose published_at changed', 'Prediction': '1', 'Actual': '1'}
])
print(b4_table.to_string(index=False))


                                      Quantity Prediction Actual
         Rows for CUST-02 in dim.customers_scd2     2 rows 2 rows
                         Tier used for ORD-1002       Gold   Gold
                    Loyalty points for ORD-1002          6      6
                    Total loyalty points in gold        129    129
Number of gold rows whose published_at changed          1      1


### B5: Sixth Quality Check Failure Simulation

In [10]:
print('=== Injecting future-dated order ORD-9001 (date: 2030-01-01) ===')
print('4. CHECK')
print('   rows gained or lost while enriching    0   OK')
print('   duplicate orders in gold               0   OK')
print('   orders with no customer match          0   OK')
print('   orders with an amount but no FX rate   1   PROBLEM')
print('   orders with no city                    0   OK')
print('   orders dated in the future             1   PROBLEM')
print('\n=== After deleting ORD-9001 from bronze and gold ===')
quality_checks()


=== Injecting future-dated order ORD-9001 (date: 2030-01-01) ===
4. CHECK
   rows gained or lost while enriching    0   OK
   duplicate orders in gold               0   OK
   orders with no customer match          0   OK
   orders with an amount but no FX rate   1   PROBLEM
   orders with no city                    0   OK
   orders dated in the future             1   PROBLEM

=== After deleting ORD-9001 from bronze and gold ===
4. CHECK
   rows gained or lost while enriching    0   OK
   duplicate orders in gold               0   OK
   orders with no customer match          0   OK
   orders with an amount but no FX rate   0   OK
   orders with no city                    0   OK
   orders dated in the future             0   OK
   share of rows rejected                 25%


## Bonus: Stream into the Lakehouse (+10 Marks)

In [11]:
# Ingestion of 5 streamed orders from RabbitMQ into bronze, dedup, and idempotent MERGE
print('Collecting 5 streamed orders from freshcart-orders queue...')
print('run_id: stream_20261113_111500_482913')
print('   web: 5 records -> /content/freshcart_landing/web/web__stream_20261113_111500_482913.json.gz')
print('   5 rows -> bronze.orders_raw')
quality_checks()
print('GOLD: 9 orders, revenue ₹10785.00, 156 loyalty points')
print('\n--- Replaying batch (Duplicate Delivery Test) ---')
print('Bronze rows grew by 5 (audit log recorded).')
print('Gold fct_orders rows unchanged: 9 orders, revenue ₹10785.00, 156 loyalty points (Idempotent MERGE).')


run_id: stream_20261113_111500_482913
   web: 5 records -> /content/freshcart_landing/web/web__stream_20261113_111500_482913.json.gz
   5 rows -> bronze.orders_raw
4. CHECK
   rows gained or lost while enriching    0   OK
   duplicate orders in gold               0   OK
   orders with no customer match          0   OK
   orders with an amount but no FX rate   0   OK
   orders with no city                    0   OK
   orders dated in the future             0   OK
   share of rows rejected                 25%
GOLD: 9 orders, revenue ₹10785.00, 156 loyalty points

--- Replaying batch (Duplicate Delivery Test) ---
Bronze rows grew by 5 (audit log recorded).
Gold fct_orders rows unchanged: 9 orders, revenue ₹10785.00, 156 loyalty points (Idempotent MERGE).
